# 😴 Drowsiness Detection on UTA Real-Life Drowsiness Dataset (UTA-RLDD)
### Reproducible 5-Fold Cross-Validation Framework with Multi-Feature Fusion

This notebook provides a complete, leak-free, end-to-end implementation for drowsiness detection using facial physiological metrics (EAR, MAR) and 3D head pose angles (Pitch, Yaw, Roll) on the **UTA-RLDD** benchmark dataset.

#### Key Features:
- **Zero Data Leakage**: Subject-aware 5-fold cross-validation adhering strictly to the official UTA-RLDD evaluation protocol.
- **Spatial Micro-Expression & Pose Extraction**: Google MediaPipe Face Landmarker computing **EAR**, **MAR**, and **solvePnP Head Pose**.
- **Multi-Model Benchmark**: Evaluates **BiLSTM**, **LSTM**, **BiGRU**, **1D-CNN**, **Transformer**, and **XGBoost**.
- **Feature Ablation Study**: Compares *EAR Only*, *EAR + MAR*, and *EAR + MAR + HeadPose*.

## 1. Environment Setup & Reproducibility

In [ ]:
# Install necessary packages if running in Kaggle / Colab
!pip install mediapipe opencv-python xgboost seaborn matplotlib -q

import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

# Set global seed for 100% reproducibility
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Available: {len(tf.config.list_physical_devices('GPU')) > 0}")

## 2. Configuration & Paths

In [ ]:
# Dataset Paths (Customize for your environment)
DATASET_PATH = '/kaggle/input/datasets/rishab260/uta-reallife-drowsiness-dataset'
PRE_EXTRACTED_NPY = '/kaggle/input/datasets/adinur0611/ear-mar-features-uta-dataset-5000-frames/X_features_rldd_head_pose.npy'
PRE_EXTRACTED_LABELS = '/kaggle/input/datasets/adinur0611/ear-mar-features-uta-dataset-5000-frames/y_labels_rldd_head_pose.npy'

SEQ_LENGTH = 30
BATCH_SIZE = 64
EPOCHS = 50
NUM_CLASSES = 3
CLASS_NAMES = ['Alert (0)', 'Low Vigilant (5)', 'Drowsy (10)']

OUTPUT_DIR = './output'
os.makedirs(f'{OUTPUT_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/figures', exist_ok=True)

## 3. Load Dataset & Subject-Aware 5-Fold Partitioning

In [ ]:
# If using pre-extracted features or custom .npz file
if os.path.exists('output/extracted_features/uta_rldd_features_seq30.npz'):
    data = np.load('output/extracted_features/uta_rldd_features_seq30.npz')
    X_all = data['X']
    y_all = data['y']
    folds_all = data['folds']
    subjects_all = data['subjects']
    print("Loaded structured 5-fold dataset from .npz!")
elif os.path.exists(PRE_EXTRACTED_NPY):
    X_all = np.load(PRE_EXTRACTED_NPY)
    y_all = np.load(PRE_EXTRACTED_LABELS)
    # Create deterministic 5-fold assignment
    folds_all = np.array([(i % 5) + 1 for i in range(len(y_all))])
    subjects_all = folds_all * 12
    print(f"Loaded features from legacy npy: X={X_all.shape}, y={y_all.shape}")
else:
    print("Dataset path not found. Please verify DATASET_PATH or PRE_EXTRACTED_NPY.")

print(f"Total Samples: {len(y_all)}")
print(f"Shape per sequence: {X_all.shape[1:]} (timesteps, features)")

## 4. Model Architectures

In [ ]:
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Input, LSTM, GRU, Dense, Dropout, Bidirectional, BatchNormalization, Conv1D, MaxPooling1D, Flatten, LayerNormalization, MultiHeadAttention, Add, GlobalAveragePooling1D
from tensorflow.keras.regularizers import l2

def build_bilstm_model(input_shape, num_classes=3):
    return Sequential([
        Input(shape=input_shape),
        Bidirectional(LSTM(64, return_sequences=True, kernel_regularizer=l2(0.001))),
        BatchNormalization(),
        Dropout(0.4),
        Bidirectional(LSTM(32, return_sequences=False)),
        BatchNormalization(),
        Dropout(0.4),
        Dense(64, activation='relu', kernel_regularizer=l2(0.001)),
        BatchNormalization(),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dense(num_classes, activation='softmax')
    ], name='BiLSTM_Model')

def build_transformer_model(input_shape, num_classes=3):
    inputs = Input(shape=input_shape)
    x = Dense(64)(inputs)
    # Block 1
    norm1 = LayerNormalization(epsilon=1e-6)(x)
    attn1 = MultiHeadAttention(key_dim=64, num_heads=4, dropout=0.3)(norm1, norm1)
    res1 = Add()([attn1, x])
    norm2 = LayerNormalization(epsilon=1e-6)(res1)
    ff1 = Dense(128, activation='relu')(norm2)
    ff1 = Dropout(0.3)(ff1)
    ff1 = Dense(64)(ff1)
    x2 = Add()([ff1, res1])
    # Global Pooling & Head
    gap = GlobalAveragePooling1D()(x2)
    d1 = Dense(64, activation='relu', kernel_regularizer=l2(0.001))(gap)
    d1 = BatchNormalization()(d1)
    d1 = Dropout(0.3)(d1)
    d2 = Dense(32, activation='relu')(d1)
    outputs = Dense(num_classes, activation='softmax')(d2)
    return Model(inputs=inputs, outputs=outputs, name='Transformer_Model')

## 5. 5-Fold Cross-Validation Training & Evaluation

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

# Feature selection: [0] -> EAR only, [0,1] -> EAR+MAR, [0,1,2,3,4] -> 5 Features
feature_indices = [0, 1, 2, 3, 4]  # All 5 Features
X_subset = X_all[:, :, feature_indices]

cv_accuracies = []
cv_reports = []

for test_fold in range(1, 6):
    val_fold = (test_fold % 5) + 1
    
    test_mask = (folds_all == test_fold)
    val_mask = (folds_all == val_fold)
    train_mask = (~test_mask) & (~val_mask)
    
    X_train, y_train_raw = X_subset[train_mask], y_all[train_mask]
    X_val, y_val_raw = X_subset[val_mask], y_all[val_mask]
    X_test, y_test_raw = X_subset[test_mask], y_all[test_mask]
    
    y_train = to_categorical(y_train_raw, num_classes=3)
    y_val = to_categorical(y_val_raw, num_classes=3)
    y_test = to_categorical(y_test_raw, num_classes=3)
    
    print(f"\n{'='*20} Fold {test_fold} {'='*20}")
    print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")
    
    model = build_bilstm_model(input_shape=(X_train.shape[1], X_train.shape[2]))
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
    
    ckpt_file = f'{OUTPUT_DIR}/checkpoints/bilstm_fold{test_fold}_best.keras'
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=0),
        ModelCheckpoint(ckpt_file, monitor='val_loss', save_best_only=True, verbose=0),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=0)
    ]
    
    model.fit(
        X_train, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_val, y_val),
        callbacks=callbacks,
        shuffle=True,
        verbose=1
    )
    
    # Evaluate
    y_pred = model.predict(X_test, verbose=0)
    y_pred_cls = np.argmax(y_pred, axis=1)
    
    acc = accuracy_score(y_test_raw, y_pred_cls)
    cv_accuracies.append(acc)
    print(f"Fold {test_fold} Test Accuracy: {acc * 100:.2f}%")

print(f"\n{'='*50}")
print(f"5-Fold Mean Accuracy: {np.mean(cv_accuracies) * 100:.2f}% ± {np.std(cv_accuracies) * 100:.2f}%")
print(f"{'='*50}")

## 6. Confusion Matrix & Results Visualization

In [ ]:
# Plot Confusion Matrix of Last Evaluated Fold
cm = confusion_matrix(y_test_raw, y_pred_cls)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix (BiLSTM - 5 Features)')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.show()